# Segmentação de Clientes (Clustering)

Este notebook foca no **ETL, Limpeza e Pré-processamento** dos dados para preparar a base para modelagem.

### O que é Clustering?
O clustering é uma técnica usada em aprendizado não supervisionado para agrupar pontos de dados semelhantes com base em suas características intrínsecas ou similaridades. Seu objetivo é identificar padrões ou estruturas nos dados sem conhecimento prévio dos agrupamentos ou rótulos.

No clustering, os pontos de dados são agrupados em clusters com base em sua proximidade uns aos outros no espaço de características. O objetivo é maximizar a similaridade dentro dos clusters e minimizar a similaridade entre clusters diferentes.

### 1. Importando Módulos e Carregando Dados

In [47]:
import pandas as pd 
import os
from datetime import datetime

# Carregando os datasets
df_transacoes = pd.read_csv('../data/df_transacoes.csv')
df_produtos = pd.read_csv('../data/df_produtos.csv')
df_vendas = pd.read_csv('../data/df_vendas.csv')
df_clientes = pd.read_csv('../data/df_clientes.csv')

## 2. ETL & Limpeza

Nesta etapa, tratamos os tipos de dados e verificamos a integridade da base.

### 2.1 Conversão de Tipos (Datas)
Garantir que colunas temporais sejam tratadas como objetos `datetime` do pandas.

In [48]:
# Convertendo colunas de data
df_vendas['data_venda'] = pd.to_datetime(df_vendas['data_venda'])
df_clientes['data_nascimento'] = pd.to_datetime(df_clientes['data_nascimento'])
df_clientes['data_cadastro'] = pd.to_datetime(df_clientes['data_cadastro'])

print("Tipos de dados após conversão:")
print(df_vendas.dtypes[['data_venda']])

Tipos de dados após conversão:
data_venda    datetime64[ns]
dtype: object


### 2.2 Verificação de Valores Nulos e Duplicatas

In [49]:
datasets = {
    "Vendas": df_vendas, 
    "Clientes": df_clientes, 
    "Produtos": df_produtos, 
    "Transações": df_transacoes
}

for nome, df in datasets.items():
    nulos = df.isnull().sum().sum()
    duplicatas = df.duplicated().sum()
    print(f"{nome:12} | Nulos: {nulos} | Duplicatas: {duplicatas}")

Vendas       | Nulos: 0 | Duplicatas: 0
Clientes     | Nulos: 0 | Duplicatas: 0
Produtos     | Nulos: 0 | Duplicatas: 0
Transações   | Nulos: 0 | Duplicatas: 0


## 3. Pré-processamento & Engenharia de Atributos

Criação de novas variáveis para facilitar a análise de agrupamento.

In [50]:
# 1. Valor Total da Venda (Quantidade * Preço)
df_vendas['valor_total'] = df_vendas['qtd_vendida'] * df_vendas['preco_praticado']

# 2. Trazer informações de produto para tabela de vendas
df_registro = pd.merge(df_vendas, df_produtos, on='produto_id', how='left')

# 3. Trazer informações de cliente para tabela de vendas    
df_registro = pd.merge(df_registro, df_clientes, on='cliente_id', how='left')

# 4. Calcule a diferença entre o preco_base e o preco_praticado em percentual ou valor absoluto
df_registro['diferenca_preco'] = df_registro['preco_base'] - df_registro['preco_praticado']
df_registro['diferenca_preco_percentual'] = (df_registro['diferenca_preco'] / df_registro['preco_base']).fillna(0)

# 5. Calcule a idade do cliente no momento da venda
df_registro['idade_cliente'] = (df_registro['data_venda'] - df_registro['data_nascimento']).dt.days // 365

# 6. Calcule o tempo desde o cadastro do cliente até a venda
df_registro['tempo_cadastro_venda'] = (df_registro['data_venda'] - df_registro['data_cadastro']).dt.days

# 7. calcular o dias desde a ultima compra do cliente se for a primeira compra, deixar como NaN ou 0    
dt_compras = df_registro.sort_values(by=['cliente_id', 'data_venda'])
df_registro['dias_desde_ultima_compra'] = df_registro.groupby('cliente_id')['data_venda'].diff().dt.days.fillna(0)

# 8. Qual item mais comprado por cliente ou qual embalagem mais comprada por cliente (ser for lapis, qual embalagem mais comprada : 12 unidades, 24 unidades, etc)
df_registro['categoria_mais_comprada'] = df_registro.groupby('cliente_id')['categoria'].transform(lambda x: x.mode()[0])

# 9. Define como 1 se o desconto foi maior que 10% do preço base
df_registro['is_promo'] = (df_registro['diferenca_preco'] / df_registro['preco_base'] > 0.10).astype(int)

# 10. Extrai o mês para captar sazonalidade (Ex: Janeiro = Volta às Aulas)
df_registro['mes_venda'] = df_registro['data_venda'].dt.month


In [51]:
df_registro.head(1)

,data_venda,cliente_id,produto_id,qtd_vendida,preco_praticado,media_preco_similares,is_final_de_semana,valor_total,nome_x,categoria,...,data_cadastro,estado,diferenca_preco,diferenca_preco_percentual,idade_cliente,tempo_cadastro_venda,dias_desde_ultima_compra,categoria_mais_comprada,is_promo,mes_venda
0,2026-03-01,CUST-1362,301,4,9.29,8.7,1,37.16,Refil Caneta Gel Pilot,Escrita,...,2024-10-14,SC,-0.29,-0.032222,63,503,0.0,Escrita,0,3


In [52]:
datasets = {
    "Vendas": df_registro, 

}
for nome, df in datasets.items():
    nulos = df.isnull().sum().sum()
    duplicatas = df.duplicated().sum()
    print(f"{nome:12} | Nulos: {nulos} | Duplicatas: {duplicatas}")

Vendas       | Nulos: 0 | Duplicatas: 0


In [53]:
df_registro.head()

,data_venda,cliente_id,produto_id,qtd_vendida,preco_praticado,media_preco_similares,is_final_de_semana,valor_total,nome_x,categoria,...,data_cadastro,estado,diferenca_preco,diferenca_preco_percentual,idade_cliente,tempo_cadastro_venda,dias_desde_ultima_compra,categoria_mais_comprada,is_promo,mes_venda
0,2026-03-01,CUST-1362,301,4,9.29,8.70,1,37.16,Refil Caneta Gel Pilot,Escrita,...,2024-10-14,SC,-0.29,-0.032222,63,503,0.0,Escrita,0,3
1,2026-03-01,CUST-1213,201,1,119.55,120.39,1,119.55,Caderno Moleskine Classic,Papelaria,...,2023-01-12,RS,0.45,0.003750,66,1144,0.0,Papelaria,0,3
2,2026-03-01,CUST-1051,103,1,2.45,2.49,1,2.45,Caneta Esferográfica Bic Cristal,Escrita,...,2025-09-19,PR,0.05,0.020000,52,163,0.0,Escrita,0,3
3,2026-03-01,CUST-1093,102,1,14.98,14.46,1,14.98,Caneta Gel Uni-ball Signo,Escrita,...,2023-05-12,PR,-0.48,-0.033103,42,1024,0.0,Escrita,0,3
4,2026-03-01,CUST-1411,103,2,2.37,2.62,1,4.74,Caneta Esferográfica Bic Cristal,Escrita,...,2023-02-07,RS,0.13,0.052000,38,1118,0.0,Papelaria,0,3


In [54]:
# Lista de colunas para remover
colunas_para_remover = [

    'nome_x', 
    'nome_y', 
    'categoria', 
    'subcategoria', 
    'marca', 
    'caracteristicas', 
    'data_cadastro', 
    'estado', 
    'preco_base',
    'qtd_vendida',
    'preco_praticado',
    'media_preco_similares',
    'is_final_de_semana'
    # Remova se não for usar One-Hot Encoding nela agora
]

# Criando o dataset de treino
df_pre_treino = df_registro.drop(columns=colunas_para_remover)

print(f"Dataset pronto! Colunas restantes: {df_pre_treino.columns.tolist()}")

Dataset pronto! Colunas restantes: ['data_venda', 'cliente_id', 'produto_id', 'valor_total', 'genero', 'data_nascimento', 'diferenca_preco', 'diferenca_preco_percentual', 'idade_cliente', 'tempo_cadastro_venda', 'dias_desde_ultima_compra', 'categoria_mais_comprada', 'is_promo', 'mes_venda']


In [55]:
df_pre_treino.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 951 entries, 0 to 950
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   data_venda                  951 non-null    datetime64[ns]
 1   cliente_id                  951 non-null    object        
 2   produto_id                  951 non-null    int64         
 3   valor_total                 951 non-null    float64       
 4   genero                      951 non-null    object        
 5   data_nascimento             951 non-null    datetime64[ns]
 6   diferenca_preco             951 non-null    float64       
 7   diferenca_preco_percentual  951 non-null    float64       
 8   idade_cliente               951 non-null    int64         
 9   tempo_cadastro_venda        951 non-null    int64         
 10  dias_desde_ultima_compra    951 non-null    float64       
 11  categoria_mais_comprada     951 non-null    object        

# Criando o dataset de treino a partir do dataset de clientes agregando as informações de vendas.

In [56]:
df_clientes_header = df_clientes.copy()
df_clientes_header = df_clientes_header[['cliente_id', 'data_nascimento','data_cadastro']].reset_index(drop=True)    

In [57]:
df_clientes_header

,cliente_id,data_nascimento,data_cadastro
0,CUST-1000,1965-11-06,2024-12-01
1,CUST-1001,1997-10-13,2024-01-08
2,CUST-1002,2006-01-28,2024-05-06
3,CUST-1003,1987-08-04,2024-04-04
4,CUST-1004,1959-11-21,2024-05-25
...,...,...,...
495,CUST-1495,1962-12-04,2023-01-01
496,CUST-1496,1963-04-25,2024-03-20
497,CUST-1497,1980-05-08,2024-03-28
498,CUST-1498,1978-06-13,2024-11-19


In [58]:
df_pre_treino.head(1)

,data_venda,cliente_id,produto_id,valor_total,genero,data_nascimento,diferenca_preco,diferenca_preco_percentual,idade_cliente,tempo_cadastro_venda,dias_desde_ultima_compra,categoria_mais_comprada,is_promo,mes_venda
0,2026-03-01,CUST-1362,301,37.16,M,1962-12-24,-0.29,-0.032222,63,503,0.0,Escrita,0,3


In [ ]:
# 1. Agrega o valor total por cliente (Gera um DF com 429 linhas)
df_agregado = df_pre_treino.groupby('cliente_id')['valor_total'].sum().reset_index()

# 1.1. Une com a tabela de cabeçalho (Garante as 500 linhas)
df_clientes_header = pd.merge(
    df_clientes_header, 
    df_agregado, 
    on='cliente_id', 
    how='left'
)

# 1.2. Renomeia e limpa os que não compraram (NaN -> 0)
df_clientes_header = df_clientes_header.rename(columns={'valor_total': 'valor_total_gasto'})
df_clientes_header['valor_total_gasto'] = df_clientes_header['valor_total_gasto'].fillna(0)

# 2 ...

In [70]:
# Preparando as agregações no dataset de registros (df_pre_treino)
df_features = df_pre_treino.groupby('cliente_id').agg({
    # V1: Média da diferença de preço (Sensibilidade)
    'diferenca_preco': 'mean',
    
    # V2: Tempo de Atividade (Fidelidade Temporal em dias)
    'data_venda': lambda x: (x.max() - x.min()).days,
    
    # V3: Propensão a Promoção (% de compras com desconto)
    'is_promo': 'mean',
    
    # V4: Categoria mais comprada (Moda)
    'categoria_mais_comprada': lambda x: x.mode()[0],

    # V5: Mes de Pico de Compras (Sazonalidade)
    'mes_venda': lambda x: x.mode()[0]

}).reset_index()

# Renomeando para clareza
df_features.columns = ['cliente_id', 'avg_desconto', 'tempo_atividade_dias', 'propensao_promo', 'perfil_categoria', 'mes_pico_compra']

# Unindo ao cabeçalho principal (Garantindo os 500 clientes)
df_clientes_header = pd.merge(df_clientes_header, df_features, on='cliente_id', how='left')

# Tratamento de nulos para quem não tem histórico de compra
df_clientes_header['avg_desconto'] = df_clientes_header['avg_desconto'].fillna(0)
df_clientes_header['tempo_atividade_dias'] = df_clientes_header['tempo_atividade_dias'].fillna(0)
df_clientes_header['propensao_promo'] = df_clientes_header['propensao_promo'].fillna(0)
df_clientes_header['perfil_categoria'] = df_clientes_header['perfil_categoria'].fillna('Sem Histórico')


# V3: Idade atualizada (baseada no cadastro)
df_clientes_header['idade'] = 2026 - pd.to_datetime(df_clientes_header['data_nascimento']).dt.year

# V5: Tempo desde o cadastro (Tenure em dias)
data_hoje = pd.to_datetime('2026-04-06')
df_clientes_header['tenure_dias'] = (data_hoje - pd.to_datetime(df_clientes_header['data_cadastro'])).dt.days

KeyError: 'perfil_categoria'

In [44]:
# 1. Calculando métricas auxiliares de frequência e sazonalidade
df_aux = df_pre_treino.groupby('cliente_id').agg({
    # V6: Intervalo Médio entre compras
    'data_venda': [
        lambda x: x.diff().dt.days.mean(), # Média de dias entre compras
        'count'                            # Total de compras para identificar compra única
    ],
    # Sazonalidade: % de compras no mês de Março (Exemplo de Volta às Aulas/Pico)
    'mes_venda': lambda x: (x == 3).mean() 
}).reset_index()

# Ajustando nomes das colunas
df_aux.columns = ['cliente_id', 'intervalo_medio_dias', 'total_compras', 'propensao_marco']

# 2. Criando o Flag de Compra Única (V6 parte 2)
df_aux['is_single_buyer'] = (df_aux['total_compras'] <= 1).astype(int)
df_aux['intervalo_medio_dias'] = df_aux['intervalo_medio_dias'].fillna(0) # Se só comprou 1x, intervalo é 0

# 3. Unindo ao dataset principal
df_clientes_header = pd.merge(df_clientes_header, df_aux, on='cliente_id', how='left')

# 4. Tratamento de nulos para quem não comprou
df_clientes_header['intervalo_medio_dias'] = df_clientes_header['intervalo_medio_dias'].fillna(0)
df_clientes_header['is_single_buyer'] = df_clientes_header['is_single_buyer'].fillna(1) # Sem compra = Compra única/Inativo
df_clientes_header['propensao_marco'] = df_clientes_header['propensao_marco'].fillna(0)

# 5. One-Hot Encoding para a Categoria (Variável #7)
# Isso transforma 'Escrita' ou 'Papelaria' em colunas de 0 e 1
df_clientes_header = pd.get_dummies(df_clientes_header, columns=['perfil_categoria'], prefix='cat')

# Removendo coluna auxiliar de contagem para evitar redundância com outras métricas de frequência
df_clientes_header = df_clientes_header.drop(columns=['total_compras'])

In [64]:
df_clientes_header.head()

,cliente_id,data_nascimento,data_cadastro,avg_desconto,tempo_atividade_dias,propensao_promo,idade,tenure_dias,intervalo_medio_dias,is_single_buyer,cat_Escrita,cat_Papelaria,cat_Sem Histórico,pico_mes_0,pico_mes_3
0,CUST-1000,1965-11-06,2024-12-01,-0.130,5.0,0.0,61,491,5.000000,0.0,True,False,False,False,True
1,CUST-1001,1997-10-13,2024-01-08,0.010,0.0,0.0,29,819,0.000000,1.0,True,False,False,False,True
2,CUST-1002,2006-01-28,2024-05-06,-0.245,13.0,0.0,20,700,13.000000,0.0,True,False,False,False,True
3,CUST-1003,1987-08-04,2024-04-04,4.590,0.0,0.0,39,732,0.000000,1.0,False,True,False,False,True
4,CUST-1004,1959-11-21,2024-05-25,0.660,17.0,0.0,67,681,5.666667,0.0,True,False,False,False,True


In [65]:
df_clientes_header.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   cliente_id            500 non-null    object        
 1   data_nascimento       500 non-null    datetime64[ns]
 2   data_cadastro         500 non-null    datetime64[ns]
 3   avg_desconto          500 non-null    float64       
 4   tempo_atividade_dias  500 non-null    float64       
 5   propensao_promo       500 non-null    float64       
 6   idade                 500 non-null    int32         
 7   tenure_dias           500 non-null    int64         
 8   intervalo_medio_dias  500 non-null    float64       
 9   is_single_buyer       500 non-null    float64       
 10  cat_Escrita           500 non-null    bool          
 11  cat_Papelaria         500 non-null    bool          
 12  cat_Sem Histórico     500 non-null    bool          
 13  pico_mes_0          

### Próximos Passos
Com os dados limpos e enriquecidos, o próximo passo é realizar a **Análise RFM** no `Notebook 02.ipynb`.